**K-Means Clustering** is a popular and simple unsupervised machine learning algorithm. It is used to find hidden groups or patterns in a dataset that does not have any target labels or predefined categories (y).

Its main goal is to partition your data points into K distinct, non-overlapping clusters based purely on their geometric similarity.

# Core Intuition

K-Means partitions an unlabeled dataset into $K$ distinct, non-overlapping clusters by placing centroids and assigning each point to its nearest center.

1. **Objective Function**
    Minimize the total squared Euclidean distance between data points $x_i$ and their assigned cluster centroids $\mu_k$:
    $$J = \sum_{k=1}^{K} \sum_{i \in C_k} \Vert{}x_i - \mu_k\Vert{}^2$$

2. **Step-by-Step Algorithm**
    1. **Initialize**: Randomly pick $K$ data points as initial cluster centroids $\mu_1, \mu_2, \dots, \mu_K$.
    2. **Assign**: Assign each data point $x_i$ to the nearest centroid using Euclidean distance:
    $$C^{(i)} = \arg\min_k \Vert{}x_i - \mu_k\Vert{}^2$$
    3. **Update**: Recalculate each centroid as the mean of all points assigned to that cluster:
    $$\mu_k = \frac{1}{\vert{}C_k\vert{}} \sum_{i \in C_k} x_i$$
    4. **Repeat**: Iterate steps 2 and 3 until centroids stop moving (convergence).

    

# Mathematical Example

1. **Dataset**

    - X = [2,4,10,12]

2. **Initialize Centroids Randomly**: 

    We choose K = 2 and pick two random numbers to act as our initial cluster centers (μ₁ and μ₂):

    - Centroid 1 (μ₁) = 3
    - Centroid 2 (μ₂) = 11

3. **Assign Points to the Closest Centroid**

    We calculate the absolute distance from every data point to both centroids. The point joins the cluster of the centroid that yields the smallest distance.

    - Point 2:
        - Distance to μ₁(3) is |2 - 3| = 1
        - Distance to μ₂(11) is |2 - 11| = 9
        - Result: Joins Cluster 1
        
    - Point 4:
        - Distance to μ₁(3) is |4 - 3| = 1
        - Distance to μ₂(11) is |4 - 11| = 7
        - Result: Joins Cluster 1
    
    - Point 10:
        - Distance to μ₁(3) is |10 - 3| = 7
        - Distance to μ₂(11) is |10 - 11| = 1
        - Result: Joins Cluster 2
        
    - Point 12:
        - Distance to μ₁(3) is |12 - 3| = 9
        - Distance to μ₂(11) is |12 - 11| = 1
        - Result: Joins Cluster 2

    Our current groupings are: Cluster 1 = ${2, 4}$ and Cluster 2 = ${10, 12}$.

4. **Update Centroids (Calculate the Mean)**

    Now, the centroids must move to the exact mathematical average (mean) of the points assigned to them.
    - New μ₁ = $\frac{2 + 4}{2} = \mathbf{3}$
    - New μ₂ = $\frac{10 + 12}{2} = \mathbf{11}$

5. **Repeat Until Convergence**
    Because the updated centroids came out to be exactly the same numbers (3 and 11), running the next iteration will yield the exact same assignments. The algorithm has converged and stops.
    
    Our final discovered clusters are safely locked in: ${2, 4}$ and ${10, 12}$.


# Crucial Concepts You Must Know

1. **The Mathematical Objective Function (Inertia)**
    
    K-Means works under the hood to minimize a metric called Inertia or the Within-Cluster Sum of Squares (WCSS). It wants to minimize the squared distance between points and their assigned centroids:
    
    $$\text{WCSS}=\sum _{j=1}^{K}\sum _{i\in C_{j}}||x_{i}-\mu _{j}||{}^{2}$$
    
2. **Feature Scaling is Mandatory**
    Just like **k-NN**, K-Means calculates straight line distances (Euclidean Distance). If one feature is massive (like Salary) and another is small (like Age), the large feature will completely swallow the small one. You **must** standardize your features using StandardScaler first.
    
3. **How do you choose the value of K? (The Elbow Method)**
    Since the data has no labels, how do you know if you need 2 clusters, 5 clusters, or 10 clusters?
    - You run K-Means multiple times with different values of K (e.g., K=1 to 10) and plot the WCSS score on a graph.
    - As K increases, WCSS will naturally drop.
    - You look for the "Elbow Point" on the line graph—the specific point where the distortion drops drastically and then flattens out. That point represents your optimal number of clusters.

# Python Implementation

In [ ]:
import numpy as np


class KMeans:

  def __init__(self, k=3, max_iters=100, tol=1e-4):
    self.k = k
    self.max_iters = max_iters
    self.tol = tol
    self.centroids = None

  def fit(self, X):
    X = np.asarray(X, dtype=np.float64)
    n_samples, _ = X.shape

    # 1. Initialize centroids randomly from data points
    np.random.seed(42)
    random_indices = np.random.choice(n_samples, self.k, replace=False)
    self.centroids = X[random_indices]

    for _ in range(self.max_iters):
      # 2. Compute pairwise distance between samples (N, 1, D) and centroids (1, K, D)
      distances = np.linalg.norm(X[:, np.newaxis] - self.centroids, axis=2)
      labels = np.argmin(distances, axis=1)

      # 3. Compute new centroids as cluster means
      new_centroids = np.array([
          X[labels == k].mean(axis=0) if np.any(labels == k) else self.centroids[k]
          for k in range(self.k)
      ])

      # 4. Convergence check
      centroid_shift = np.linalg.norm(self.centroids - new_centroids)
      self.centroids = new_centroids

      if centroid_shift < self.tol:
        break

    return self

  def predict(self, X):
    X = np.asarray(X, dtype=np.float64)
    distances = np.linalg.norm(X[:, np.newaxis] - self.centroids, axis=2)
    return np.argmin(distances, axis=1)


# TEST SCRIPT

# Synthetic 2D Dataset with 3 clear clusters
X_train = np.vstack([
    np.random.normal(loc=[2, 2], scale=0.5, size=(20, 2)),
    np.random.normal(loc=[8, 8], scale=0.5, size=(20, 2)),
    np.random.normal(loc=[2, 8], scale=0.5, size=(20, 2)),
])

# Instantiate & Fit
kmeans = KMeans(k=3)
kmeans.fit(X_train)

# Query Test Points
X_test = np.array([[2.1, 1.9], [7.9, 8.1], [1.8, 8.2]])
predictions = kmeans.predict(X_test)

print("=== K-MEANS CLUSTERING RESULTS ===")
print("Final Centroids:\n", np.round(kmeans.centroids, 2))
for i, sample in enumerate(X_test):
  print(f"Sample {sample} -> Assigned Cluster: {predictions[i]}")

=== K-MEANS CLUSTERING RESULTS ===
Final Centroids:
 [[2.06 7.91]
 [1.97 1.98]
 [8.13 7.98]]
Sample [2.1 1.9] -> Assigned Cluster: 1
Sample [7.9 8.1] -> Assigned Cluster: 2
Sample [1.8 8.2] -> Assigned Cluster: 0
